# Plot Visualization for Number of Users against Metrics
### Load the files from scripts/ directory

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import pickle
from sklearn.metrics import mean_absolute_error

# 1. Find the project root (assuming notebook is in a subfolder like /notebooks/)
PROJECT_ROOT = os.path.abspath("..")

# 2. Define the exact paths to your custom code
scripts_dir = os.path.join(PROJECT_ROOT, 'scripts')
dl_dir = os.path.join(scripts_dir, 'deep_learning_4')

# 3. Add ALL of them to Python's "Address Book" (sys.path)
for path in [PROJECT_ROOT, scripts_dir, dl_dir]:
    if path not in sys.path:
        sys.path.append(path)

# 4. Now import your custom modules
from ml_config import MLConfig # pyright: ignore[reportMissingImports]
from threshold_selection import ThresholdSelector # pyright: ignore[reportMissingImports]
from online_selector import OnlineModeSelector # pyright: ignore[reportMissingImports]

print("✅ Local VS Code Environment Ready for Visualization!")
print(f"📁 Project Root: {PROJECT_ROOT}")

✅ Local VS Code Environment Ready for Visualization!
📁 Project Root: c:\Users\Zul Arif Nael\OneDrive\Documents\EE UM Courses\Final Year Project\FYP Codebase


In [ ]:
# 🛑 SET TARGET FOLDER HERE BEFORE RUNNING:
EXPERIMENT_FOLDER = 'ablation_win16_kdeTrue_AR' 
DATASET = 'preprocessed_proposal'
MODEL_TO_PLOT = 'dnn' 
MODE = 'd2d'

print(f"Loading {MODEL_TO_PLOT.upper()} from {EXPERIMENT_FOLDER}...")

base_path = os.path.join(PROJECT_ROOT, "data", DATASET, MODE)

# 1. Load Data & Scalers
X_test = np.load(os.path.join(base_path, "X_test.npy"))
y_test = np.load(os.path.join(base_path, "y_test.npy"))

with open(os.path.join(base_path, "scaler.pkl"), "rb") as f:
    feature_scaler = pickle.load(f)
with open(os.path.join(base_path, "target_scaler.pkl"), "rb") as f:
    target_scaler = pickle.load(f)

# 2. Intercept Keras 3 formats for Local VS Code compatibility
class PatchedDense(tf.keras.layers.Dense):
    def __init__(self, **kwargs):
        kwargs.pop('quantization_config', None)
        super().__init__(**kwargs)

model_path = os.path.join(PROJECT_ROOT, "models", DATASET, EXPERIMENT_FOLDER, MODE, MODEL_TO_PLOT, f"{MODEL_TO_PLOT}_model.keras")

if not os.path.exists(model_path):
    raise FileNotFoundError(f"❌ Cannot find model at: {model_path}\nDid you download the models folder from Drive?")

# 3. Load Model using the Patch
with tf.keras.utils.custom_object_scope({'Dense': PatchedDense}):
    model = tf.keras.models.load_model(model_path)

# 4. Format Input for Windowed Models
if MODEL_TO_PLOT in ['cnn', 'dnn']:
    window_size = MLConfig.WINDOW_SIZE
    X_win = []
    for i in range(X_test.shape[0]):
        for t in range(X_test.shape[1] - window_size + 1):
            X_win.append(X_test[i, t:t+window_size, :])
    X_pred_input = np.array(X_win)
else:
    X_pred_input = X_test

print("Running Predictions...")
y_pred_scaled = model.predict(X_pred_input, batch_size=64, verbose=0)

# 5. Inverse Transform Targets to Raw dB
y_pred_db = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

if MODEL_TO_PLOT in ['cnn', 'dnn']:
    y_win = []
    for i in range(X_test.shape[0]):
        for t in range(X_test.shape[1] - window_size + 1):
            y_win.append(y_test[i, t+window_size-1, 0])
    y_test_aligned = np.array(y_win)
else:
    y_test_aligned = y_test.flatten()

y_true_db = target_scaler.inverse_transform(y_test_aligned.reshape(-1, 1)).flatten()

print("✅ Data Un-scaled and Predictions Generated!")

Loading DNN from ablation_win16_kdeTrue_AR...


TypeError: <class 'keras.src.models.sequential.Sequential'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras', 'class_name': 'Sequential', 'config': {'name': 'sequential', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None, 'shared_object_id': 136724756206544}, 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_shape': [None, 16, 19], 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_layer', 'optional': False}, 'registered_name': None}, {'module': 'keras.layers', 'class_name': 'Flatten', 'config': {'name': 'flatten', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'data_format': 'c

In [ ]:
# ==========================================
# CELL 3: EPISODE METRICS AGGREGATION
# ==========================================
# Based on preprocess_data.py, 'num_interferers' is the 6th feature (index 5)
NUM_INTERF_INDEX = 5 

results_list = []
total_windows_per_ep = X_test.shape[1] - MLConfig.WINDOW_SIZE + 1 if MODEL_TO_PLOT in ['cnn', 'dnn'] else X_test.shape[1]

# Loop through each episode
for ep_idx in range(X_test.shape[0]):
    # Extract the number of interferers for this specific episode
    sample_timestep_scaled = X_test[ep_idx, 0, :].reshape(1, -1)
    sample_unscaled = feature_scaler.inverse_transform(sample_timestep_scaled)
    interferers = int(round(sample_unscaled[0][NUM_INTERF_INDEX]))
    
    # Extract the true and predicted SINR just for this episode
    start_idx = ep_idx * total_windows_per_ep
    end_idx = start_idx + total_windows_per_ep
    
    ep_y_true = y_true_db[start_idx:end_idx]
    ep_y_pred = y_pred_db[start_idx:end_idx]
    
    # Calculate MAE for this episode
    ep_mae = mean_absolute_error(ep_y_true, ep_y_pred)
    
    # Calculate Shannon capacity
    ts = ThresholdSelector(bandwidth_hz=MLConfig.BANDWIDTH_HZ)
    ep_tput_array = [ts.shannon_throughput(sinr) for sinr in ep_y_true]
    ep_avg_tput = np.mean(ep_tput_array)
    
    results_list.append({
        'num_interferers': interferers,
        'mae': ep_mae,
        'throughput_mbps': ep_avg_tput
    })

df_results = pd.DataFrame(results_list)

# Group the data to calculate Mean and Standard Deviation
grouped_stats = df_results.groupby('num_interferers').agg(
    MAE_mean=('mae', 'mean'),
    MAE_std=('mae', 'std'),
    Tput_mean=('throughput_mbps', 'mean'),
    Tput_std=('throughput_mbps', 'std')
).reset_index()

print("✅ Grouped Statistics Calculated:")
print(grouped_stats)

In [ ]:
# ==========================================
# CELL 4: PLOT ERROR ANALYSIS
# ==========================================
plt.figure(figsize=(10, 6))

x = grouped_stats['num_interferers']
y_mean = grouped_stats['MAE_mean']
y_std = grouped_stats['MAE_std']

# Plot the solid mean line
plt.plot(x, y_mean, color='red', marker='o', linewidth=2, label=f'{MODEL_TO_PLOT.upper()} Mean MAE')

# Shade the Standard Deviation area (+/- 1 Std Dev)
plt.fill_between(x, y_mean - y_std, y_mean + y_std, color='red', alpha=0.2, label='±1 Standard Deviation')

plt.title('Prediction Error vs. Number of Interfering Devices', fontsize=14, fontweight='bold')
plt.xlabel('Number of Interfering Devices', fontsize=12)
plt.ylabel('Mean Absolute Error (dB)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper left', fontsize=11)

plt.xticks(x)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# CELL 5: PLOT THROUGHPUT VARIATIONS
# ==========================================
plt.figure(figsize=(10, 6))

x = grouped_stats['num_interferers']
y_mean = grouped_stats['Tput_mean']
y_std = grouped_stats['Tput_std']

# Plot the solid mean line
plt.plot(x, y_mean, color='blue', marker='s', linewidth=2, label='Mean Average Throughput')

# Shade the Standard Deviation area
plt.fill_between(x, y_mean - y_std, y_mean + y_std, color='blue', alpha=0.2, label='±1 Standard Deviation')

plt.title('D2D Channel Capacity vs. Number of Interfering Devices', fontsize=14, fontweight='bold')
plt.xlabel('Number of Interfering Devices', fontsize=12)
plt.ylabel('Average Throughput (Mbps)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper right', fontsize=11)

plt.xticks(x)
plt.tight_layout()
plt.show()